# 01 - Exploratory Analysis

Sprint 1 - Data Understanding

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_PATH = Path('../data/raw')

customers = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
orders = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
payments = pd.read_csv(DATA_PATH / 'olist_order_payments_dataset.csv')

## Dataset Overview

In [3]:
overview = pd.DataFrame({
    'dataset': ['customers', 'orders', 'order_items', 'products', 'payments'],
    'rows': [len(customers), len(orders), len(order_items), len(products), len(payments)],
    'columns': [customers.shape[1], orders.shape[1], order_items.shape[1], products.shape[1], payments.shape[1]]
})

overview

,dataset,rows,columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,products,32951,9
4,payments,103886,5


## Data Quality Assessment

In [4]:
for name, df in {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'payments': payments
}.items():

    print(f'\n{name.upper()}')
    print(df.isnull().sum().sort_values(ascending=False).head(10))


CUSTOMERS
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

ORDERS
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

ORDER_ITEMS
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

PRODUCTS
product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm               2
product_length_cm               2
product_width_cm                2
product_id        

In [5]:
for name, df in {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'payments': payments
}.items():

    print(f'{name}: {df.duplicated().sum()} duplicated rows')

customers: 0 duplicated rows
orders: 0 duplicated rows
order_items: 0 duplicated rows
products: 0 duplicated rows
payments: 0 duplicated rows


## Business Overview

In [6]:
customers['customer_unique_id'].nunique()

96096

In [7]:
orders['order_id'].nunique()

99441

In [8]:
products['product_id'].nunique()

32951

## Data Coverage

In [9]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

In [10]:
orders['order_purchase_timestamp'].min()

Timestamp('2016-09-04 21:15:19')

In [11]:
orders['order_purchase_timestamp'].max()

Timestamp('2018-10-17 17:30:18')

## Revenue

In [12]:
revenue = order_items['price'].sum()

print(f'Revenue: {revenue:,.2f}')

Revenue: 13,591,643.70


## Repeat Customers

In [13]:
orders_customers = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

In [14]:
customer_orders = (
    orders_customers
    .groupby('customer_unique_id')
    .agg(orders=('order_id', 'nunique'))
)

In [15]:
repeat_customers = (
    customer_orders['orders'] > 1
).mean()

repeat_customers

np.float64(0.031187562437562436)

## Monthly Revenue

In [16]:
sales = (
    order_items
    .merge(
        orders[['order_id', 'order_purchase_timestamp']],
        on='order_id'
    )
)

In [17]:
sales['month'] = (
    sales['order_purchase_timestamp']
    .dt.to_period('M')
)

In [21]:
monthly_revenue = (
    sales
    .groupby('month')
    .agg(revenue=('price', 'sum'))
    .reset_index()
)

monthly_revenue.head(50)

,month,revenue
0,2016-09,267.36
1,2016-10,49507.66
2,2016-12,10.90
3,2017-01,120312.87
4,2017-02,247303.02
5,2017-03,374344.30
6,2017-04,359927.23
7,2017-05,506071.14
8,2017-06,433038.60
9,2017-07,498031.48
